# Bayesian MVP — Applied to Orbitrap HILIC posESI

**Goal:** Test how well the negESI-trained model generalizes to posESI data (same platform, different polarity).

**Input:** `data/Orbitrap_HILIC_posESI_curated_041326.csv` (13,608 spectra, Oliver's curation)

**Approach:**
1. Retrieve full library hit pool for pos spectra from MassWiki
2. Train Bayesian scoring model on **negESI** data (reproduce the 91.1% holdout result)
3. **Freeze** the fitted parameters
4. Apply to posESI — evaluate top-1 accuracy on pos TPs, FP behavior, distribution shifts

**Labels in pos data (extracted from `name`/`annotation-name`):**
- TP: `is_manual_annotated=True`, normal name (~3,024)
- FP: `yy_` prefix (714)
- TN: `zz_` prefix (135)
- Unlabeled/unannotated: blank name (~9,735)


In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
from scipy.stats import norm, gaussian_kde
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt

try:
    from rdkit import Chem, RDLogger
    from rdkit.Chem.inchi import MolToInchi, InchiToInchiKey
    RDLogger.DisableLog('rdApp.*')
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print('WARNING: RDKit not available')

# ── Paths ─────────────────────────────────────────────────────────
ROOT = '/Users/ellayoung/Desktop/metabolo_confi_score'
POS_CSV        = f'{ROOT}/data/Orbitrap_HILIC_posESI_curated_041326.csv'
POS_HITS       = f'{ROOT}/data/library_hits/orbitrap_hilic_pos_masswiki_hits.csv'

NEG_XLSX       = f'{ROOT}/data/masswiki_Orbitrap HILIC negESI_2026-03-19.xlsx'
NEG_HITS       = f'{ROOT}/data/orbitrap_hits_refetched.csv'
SOLID_TP_PATH  = f'{ROOT}/data/solid_tp.csv'
PRIOR_CACHE    = f'{ROOT}/data/pubmed_prior_cache.json'
INCHIKEY_CACHE = f'{ROOT}/data/inchikey_cache.json'

OUT_DIR        = f'{ROOT}/results/orbitrap_pos_test'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Constants from neg model ─────────────────────────────────────
SIGMA_BROAD    = 10.0
SIGMA_RT_BROAD = 300.0
P_MS1_NULL     = norm.pdf(0, 0, SIGMA_BROAD)
P_RT_NULL      = norm.pdf(0, 0, SIGMA_RT_BROAD)
MS2_MIN        = 0.75
NOTA_THRESH    = 0.6
PRIOR_FLOOR    = 0.01

print(f'P_MS1_NULL = {P_MS1_NULL:.6f}')
print(f'P_RT_NULL  = {P_RT_NULL:.6f}')

## 1. Load & Label Pos Data

In [ ]:
# ── Load pos CSV ──────────────────────────────────────────────────
pos_raw = pd.read_csv(POS_CSV, low_memory=False)
print(f'Loaded {len(pos_raw):,} pos spectra, {pos_raw.shape[1]} columns')

# ── Normalize column names: the pos CSV uses some different conventions
# Standardize to match the neg pipeline (which expects: name, smiles, adduct)
# In pos CSV, 'name' and 'annotation-name' are essentially the same content
# (both contain Oliver's annotations), so we use 'name' as canonical.

# ── Label extraction ─────────────────────────────────────────────
pos_raw['label'] = 'unlabeled'
names = pos_raw['name'].fillna('').astype(str)
yy_mask = names.str.startswith('yy_')
zz_mask = names.str.startswith('zz_')
# TP: is_manual_annotated AND not yy_/zz_ AND not blank
tp_mask = pos_raw['is_manual_annotated'].astype(bool) & ~yy_mask & ~zz_mask & (names.str.strip() != '')
pos_raw.loc[tp_mask, 'label'] = 'TP'
pos_raw.loc[yy_mask, 'label'] = 'FP'
pos_raw.loc[zz_mask, 'label'] = 'TN'

print(f'\nPos label distribution:')
print(pos_raw['label'].value_counts().to_string())

# ── Pos annotated subset (TP + FP only, exclude TN and unlabeled) ──
pos = pos_raw[pos_raw['label'].isin(['TP', 'FP'])].copy()
print(f'\nAnnotated pos subset (TP + FP): {len(pos):,} spectra')

## 2. Retrieve Library Hits

The pos spreadsheet includes only top-3 reference + top-1 annotation hit inline.
For proper Bayesian ranking we need the full candidate pool (~40+ hits/spectrum) from MassWiki.

This cell checks if the hits file exists; if not, it prints fetch instructions.

In [ ]:
if os.path.exists(POS_HITS):
    pos_hits_raw = pd.read_csv(POS_HITS, low_memory=False)
    print(f'Loaded pos hits: {len(pos_hits_raw):,} rows, {pos_hits_raw["wiki_id"].nunique():,} spectra')
else:
    print('Pos hits file not found. Run the fetch script:')
    print(f'  python code/fetch_orbitrap_pos_hits.py <MASSWIKI_TOKEN>')
    print(f'Expected output: {POS_HITS}')
    print('\nEstimated fetch time: ~40 minutes for 13,608 spectra at 6 RPS.')
    raise SystemExit('Fetch pos hits first, then re-run from this cell.')

pos_hits_raw = pos_hits_raw.rename(columns={'id': 'library_id', 'lib_name': 'name'})
print(f'Columns: {list(pos_hits_raw.columns)}')

## 3. Train Neg Model

Reproduce the orbitrap_mvp.ipynb pipeline on negESI data to obtain:
- KDE-based LR functions for MS2 similarity and sim_gap
- Scalar params: σ_M, σ_RT_anno, σ_RT_ref, α, P_novel
- Per-InChIKey prior scores

In [ ]:
# ── Load neg data ──────────────────────────────────────────────
neg_raw = pd.read_excel(NEG_XLSX, header=4)
neg_raw['label'] = 'unlabeled'
neg_raw.loc[:1297, 'label'] = 'TP'
neg_raw.loc[neg_raw['name'].str.startswith('yy_', na=False), 'label'] = 'FP'
neg_raw.loc[neg_raw['name'].str.startswith('zz_', na=False), 'label'] = 'TN'

solid_tp = pd.read_csv(SOLID_TP_PATH)
holdout_wids = set(solid_tp['wiki_id'])
neg_raw['holdout'] = neg_raw['wiki_id'].isin(holdout_wids)

neg = neg_raw[neg_raw['label'].isin(['TP', 'FP'])].copy()

print(f'Neg labels:', neg_raw['label'].value_counts().to_dict())
print(f'Neg annotated subset: {len(neg):,} spectra  (holdout={neg["holdout"].sum()})')

In [ ]:
# ── InChIKey helpers (shared cache) ───────────────────────────
ik_cache = {}
if os.path.exists(INCHIKEY_CACHE):
    with open(INCHIKEY_CACHE) as f:
        ik_cache = json.load(f)
    print(f'InChIKey cache loaded: {len(ik_cache):,} entries')

def smiles_to_ik14(smiles):
    if not RDKIT_AVAILABLE or not isinstance(smiles, str) or not smiles.strip():
        return None
    if smiles in ik_cache:
        return ik_cache[smiles].get('ik14')
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    ik = InchiToInchiKey(MolToInchi(mol))
    if ik:
        ik_cache[smiles] = {'ik14': ik[:14], 'inchikey': ik}
        return ik[:14]
    return None

def smiles_to_inchikey(smiles):
    if not RDKIT_AVAILABLE or not isinstance(smiles, str) or not smiles.strip():
        return None
    if smiles in ik_cache:
        return ik_cache[smiles].get('inchikey')
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    ik = InchiToInchiKey(MolToInchi(mol))
    if ik:
        ik_cache[smiles] = {'ik14': ik[:14], 'inchikey': ik}
    return ik

In [ ]:
# ── Load neg hits, compute InChIKey ───────────────────────────
neg_hits = pd.read_csv(NEG_HITS, low_memory=False)
neg_hits = neg_hits.rename(columns={'id': 'library_id', 'lib_name': 'name'})
print(f'Neg hits: {len(neg_hits):,}')

if RDKIT_AVAILABLE and 'smiles' in neg_hits.columns:
    neg_hits['ik14']     = neg_hits['smiles'].apply(smiles_to_ik14)
    neg_hits['inchikey'] = neg_hits['smiles'].apply(smiles_to_inchikey)
    with open(INCHIKEY_CACHE, 'w') as f:
        json.dump(ik_cache, f)

# Build feature table for neg
def build_joint(spectra_df, hits_df):
    '''Join hits to spectra, compute delta_ppm/delta_rt/correct.'''
    spec_cols = spectra_df[['wiki_id','rt','precursor_mz','entropy','name','smiles','label']].copy()
    if 'holdout' in spectra_df.columns:
        spec_cols['holdout'] = spectra_df['holdout']
    spec_cols = spec_cols.rename(columns={'name':'obs_name','smiles':'obs_smiles'})
    spec_cols['obs_ik14'] = spec_cols['obs_smiles'].apply(smiles_to_ik14) if RDKIT_AVAILABLE else None

    j = hits_df.merge(spec_cols, on='wiki_id', how='inner')
    j['lib_precursor_mz'] = pd.to_numeric(j['lib_precursor_mz'], errors='coerce')
    j['delta_ppm'] = (j['precursor_mz'] - j['lib_precursor_mz']) / j['lib_precursor_mz'] * 1e6

    j['delta_rt'] = np.nan
    anno = j['hit_source'] == 'annotation'
    ref  = j['hit_source'] == 'reference'
    if 'anno_delta_rt' in j.columns:
        j.loc[anno, 'delta_rt'] = pd.to_numeric(j.loc[anno, 'anno_delta_rt'], errors='coerce')
    if 'predicted_rt_hilic' in j.columns:
        j.loc[ref, 'delta_rt'] = (j.loc[ref, 'rt'] -
                                   pd.to_numeric(j.loc[ref, 'predicted_rt_hilic'], errors='coerce'))

    # Correctness
    if j['obs_ik14'].notna().any() and j['ik14'].notna().any():
        j['correct'] = j['ik14'].notna() & j['obs_ik14'].notna() & (j['ik14'] == j['obs_ik14'])
        name_fb = j['name'].str.strip().str.lower() == j['obs_name'].fillna('').str.strip().str.lower()
        j['correct'] = j['correct'] | (j['ik14'].isna() & name_fb)
    else:
        j['correct'] = j['name'].str.strip().str.lower() == j['obs_name'].fillna('').str.strip().str.lower()

    has_match = j.groupby('wiki_id')['correct'].any()
    j['has_library_match'] = j['wiki_id'].map(has_match)

    # Dedup: keep max entropy_similarity per (wiki_id, lib_name)
    j = j.sort_values('entropy_similarity', ascending=False).drop_duplicates(
        subset=['wiki_id', 'name']).reset_index(drop=True)
    return j

neg_joint = build_joint(neg, neg_hits)
print(f'Neg joint: {len(neg_joint):,} rows, {neg_joint["wiki_id"].nunique():,} spectra')
print(f'  correct hits: {neg_joint["correct"].sum():,}')

In [ ]:
# ── Filter by MS2_MIN, compute sim_gap ─────────────────────────
def add_sim_gap(j, ms2_min):
    j = j[j['entropy_similarity'] >= ms2_min].copy()
    j = j.sort_values(['wiki_id','entropy_similarity'], ascending=[True,False])
    nxt = j.groupby('wiki_id')['entropy_similarity'].shift(-1)
    j['sim_gap'] = (j['entropy_similarity'] - nxt.fillna(j['entropy_similarity'])).clip(lower=0.0)
    return j

neg_joint_f = add_sim_gap(neg_joint, MS2_MIN)
print(f'Neg after entropy >= {MS2_MIN}: {len(neg_joint_f):,} rows, {neg_joint_f["wiki_id"].nunique():,} spectra')

# Train subset: TP (not holdout) + FP
neg_train = neg_joint_f[((neg_joint_f['label']=='TP') & ~neg_joint_f['holdout']) |
                        (neg_joint_f['label']=='FP')].copy()
solvable = neg_train[(neg_train['label']=='TP') & neg_train['has_library_match']]
correct_hits   = solvable[solvable['correct']]
incorrect_hits = solvable[~solvable['correct']]
print(f'Train solvable TP spectra: {solvable["wiki_id"].nunique():,}')
print(f'  correct_hits: {len(correct_hits):,}  incorrect_hits: {len(incorrect_hits):,}')

In [ ]:
# ── Fit MS2 KDE ───────────────────────────────────────────────
tp_sims = correct_hits['entropy_similarity'].dropna().values
fp_sims = incorrect_hits['entropy_similarity'].dropna().values

kde_tp = gaussian_kde(tp_sims, bw_method=0.05)
kde_fp = gaussian_kde(fp_sims, bw_method=0.05)

sim_grid = np.linspace(0.0, 1.0, 2000)
lr_grid  = np.clip(kde_tp(sim_grid) / (kde_fp(sim_grid) + 1e-10), 0.0, 1000.0)

def null_adjusted_ms2_lr(esim_arr, H, alpha_):
    lam    = 1.0 - np.exp(-alpha_ * H)
    abs_lr = np.interp(esim_arr, sim_grid, lr_grid)
    return lam * abs_lr + (1.0 - lam) * 1.0

print(f'MS2 KDE fit: TP n={len(tp_sims):,} (mean={tp_sims.mean():.3f}), FP n={len(fp_sims):,} (mean={fp_sims.mean():.3f})')

In [ ]:
# ── Fit sim_gap KDE ───────────────────────────────────────────
SCALE_GAP = 100.0
gap_tp_t = np.log1p(correct_hits['sim_gap'].fillna(0.0).values * SCALE_GAP)
gap_fp_t = np.log1p(incorrect_hits['sim_gap'].fillna(0.0).values * SCALE_GAP)

kde_gap_tp = gaussian_kde(gap_tp_t, bw_method=0.15)
kde_gap_fp = gaussian_kde(gap_fp_t, bw_method=0.15)

gap_grid   = np.linspace(0.0, 1.0, 1000)
gap_grid_t = np.log1p(gap_grid * SCALE_GAP)
lr_gap_grid = np.clip(kde_gap_tp(gap_grid_t) / (kde_gap_fp(gap_grid_t) + 1e-10), 0.0, 1000.0)

def sim_gap_lr(gap_arr):
    return np.interp(np.clip(gap_arr, 0.0, 1.0), gap_grid, lr_gap_grid)

print(f'sim_gap LR: gap=0 → {sim_gap_lr(np.array([0.0]))[0]:.3f}, gap=0.05 → {sim_gap_lr(np.array([0.05]))[0]:.3f}, gap=0.20 → {sim_gap_lr(np.array([0.20]))[0]:.3f}')

In [ ]:
# ── Fit σ_M and σ_RT ─────────────────────────────────────────
ppm_trimmed = correct_hits['delta_ppm'][correct_hits['delta_ppm'].abs() <= 5]
sigma_M = float(ppm_trimmed.std(ddof=1))
print(f'σ_M (neg) = {sigma_M:.4f} ppm  (n={len(ppm_trimmed):,})')

def fit_sigma_rt(hits_df, source, trim):
    m = (hits_df['hit_source']==source) & hits_df['correct'] & hits_df['delta_rt'].notna()
    vals = hits_df.loc[m, 'delta_rt']
    trimmed = vals[vals.abs() <= trim]
    if len(trimmed) >= 10:
        return float(trimmed.std())
    return 5.0 if source == 'annotation' else 30.0

sigma_RT_anno = fit_sigma_rt(correct_hits, 'annotation', trim=60)
sigma_RT_ref  = fit_sigma_rt(correct_hits, 'reference',  trim=120)
print(f'σ_RT_anno = {sigma_RT_anno:.1f}s, σ_RT_ref = {sigma_RT_ref:.1f}s')

In [ ]:
# ── PubMed/patent prior ────────────────────────────────────────
prior_cache = {}
if os.path.exists(PRIOR_CACHE):
    with open(PRIOR_CACHE) as f:
        prior_cache = json.load(f)
    print(f'Prior cache loaded: {len(prior_cache):,} entries')

def apply_priors(j):
    j = j.copy()
    j['prior_score'] = j['inchikey'].apply(
        lambda ik: prior_cache.get(ik, 0.5) if isinstance(ik, str) else 0.5
    )
    return j

neg_joint_f = apply_priors(neg_joint_f)
n_matched = (neg_joint_f['prior_score'] != 0.5).sum()
print(f'Neg prior non-floor: {n_matched:,}/{len(neg_joint_f):,}')

In [ ]:
# ── P_novel estimate ──────────────────────────────────────────
all_wids = set(neg['wiki_id'])
hit_wids = set(neg_joint_f['wiki_id'])
p_novel_est = len(all_wids - hit_wids) / len(all_wids)
print(f'P_novel (neg) = {p_novel_est:.4f}')

In [ ]:
# ── Fit α via MLE on solvable train TP ─────────────────────────
def log_lik_alpha(alpha, df_conf, sigma_M_, p_novel_):
    total = 0.0
    for wid, g in df_conf.groupby('wiki_id'):
        if not g['correct'].any():
            continue
        H = float(g['entropy'].iloc[0])
        LR_ms2 = null_adjusted_ms2_lr(g['entropy_similarity'].values, H, alpha)
        p_ms1  = norm.pdf(g['delta_ppm'].values, 0.0, sigma_M_)
        LR_ms1 = p_ms1 / P_MS1_NULL
        LR_gap = sim_gap_lr(g['sim_gap'].fillna(0.0).values)
        raw    = g['prior_score'].fillna(PRIOR_FLOOR).values
        priors = raw * (1.0 - p_novel_) / (raw.sum() + 1e-300)
        unnorm = priors * LR_ms2 * LR_ms1 * LR_gap
        tot    = unnorm.sum() + p_novel_
        if tot < 1e-300:
            continue
        total += np.log(unnorm[g['correct'].values].sum() / tot + 1e-300)
    return -total

solvable_fit = neg_joint_f[(neg_joint_f['label']=='TP') &
                            neg_joint_f['has_library_match'] &
                            ~neg_joint_f['holdout']].copy()
print(f'Fitting alpha on {solvable_fit["wiki_id"].nunique():,} solvable TP spectra...')

res = minimize_scalar(lambda a: log_lik_alpha(a, solvable_fit, sigma_M, p_novel_est),
                      bounds=(0.01, 20.0), method='bounded')
alpha_fitted = res.x
print(f'α* = {alpha_fitted:.4f}')

## 4. Shared Scoring Function

Uses the closures (`null_adjusted_ms2_lr`, `sim_gap_lr`) and params from above.
Will be applied to both neg (sanity check) and pos (main test).

In [ ]:
def score_group(g, sigma_M_, alpha_, p_novel_, sig_rt_anno, sig_rt_ref):
    H = float(g['entropy'].iloc[0])
    N = len(g)
    LR_ms2 = null_adjusted_ms2_lr(g['entropy_similarity'].values, H, alpha_)
    LR_ms1 = norm.pdf(g['delta_ppm'].values, 0.0, sigma_M_) / P_MS1_NULL

    LR_rt   = np.ones(N)
    has_rt  = g['delta_rt'].notna().values
    is_anno = (g['hit_source']=='annotation').values
    is_ref  = (g['hit_source']=='reference').values
    drt     = g['delta_rt'].fillna(0.0).values
    if sig_rt_anno is not None:
        m = is_anno & has_rt
        if m.any(): LR_rt[m] = norm.pdf(drt[m], 0.0, sig_rt_anno) / P_RT_NULL
    if sig_rt_ref is not None:
        m = is_ref & has_rt
        if m.any(): LR_rt[m] = norm.pdf(drt[m], 0.0, sig_rt_ref) / P_RT_NULL

    LR_gap = sim_gap_lr(g['sim_gap'].fillna(0.0).values)
    raw    = g['prior_score'].fillna(PRIOR_FLOOR).values
    priors = raw * (1.0 - p_novel_) / (raw.sum() + 1e-300)
    unnorm = priors * LR_ms2 * LR_ms1 * LR_rt * LR_gap
    tot    = unnorm.sum() + p_novel_
    if tot < 1e-300:
        post, p_nova = np.full(N, 1.0/N), p_novel_
    else:
        post, p_nova = unnorm / tot, p_novel_ / tot

    out = g.copy()
    out['LR_ms2']=LR_ms2; out['LR_ms1']=LR_ms1; out['LR_rt']=LR_rt; out['LR_gap']=LR_gap
    out['prior']=priors; out['post']=post; out['P_novel']=p_nova; out['PEP']=1.0-post
    return out

def score_all(joint_filt):
    parts = [score_group(g, sigma_M, alpha_fitted, p_novel_est, sigma_RT_anno, sigma_RT_ref)
             for _, g in joint_filt.groupby('wiki_id')]
    return pd.concat(parts, ignore_index=True)

def top_calls(scored):
    rows = []
    for wid, g in scored.groupby('wiki_id'):
        pn = g['P_novel'].iloc[0]
        label = g['label'].iloc[0]
        hold  = g.get('holdout', pd.Series([False])).iloc[0]
        if pn >= NOTA_THRESH:
            rows.append({'wiki_id':wid,'lib_name_top':None,'post_top':None,
                         'P_novel':pn,'abstain':True,'reason':'NoTA',
                         'label':label,'holdout':hold})
        else:
            t = g.loc[g['post'].idxmax()]
            rows.append({'wiki_id':wid,'lib_name_top':t['name'],'post_top':t['post'],
                         'P_novel':pn,'abstain':False,'reason':'called',
                         'label':label,'holdout':hold,
                         'correct':t.get('correct', None)})
    return pd.DataFrame(rows)

## 5. Sanity Check: Reproduce Neg 91.1% Holdout

If this doesn't hit ~91%, something is off in the re-fit and pos results won't be reliable.

In [ ]:
neg_scored = score_all(neg_joint_f)
neg_top    = top_calls(neg_scored)

test_calls = neg_top[neg_top['holdout']]
called = test_calls[~test_calls['abstain']]
acc = called['correct'].mean() if len(called) else 0.0
print(f'Neg holdout (solid TP, n={len(test_calls)}):')
print(f'  Called: {len(called)}  Abstained: {(test_calls["abstain"]).sum()}')
print(f'  Top-1 accuracy: {acc:.3f}')
print(f'  Target (from orbitrap_mvp.ipynb): 0.911')

# Baseline
baseline = (neg_joint_f[neg_joint_f['holdout']]
            .sort_values('entropy_similarity', ascending=False)
            .drop_duplicates('wiki_id')['correct'].mean())
print(f'  Baseline (top entropy_similarity): {baseline:.3f}')

print(f'\n════ FITTED PARAMS (neg) ════')
print(f'σ_M            = {sigma_M:.4f} ppm')
print(f'σ_RT_anno      = {sigma_RT_anno:.1f}s')
print(f'σ_RT_ref       = {sigma_RT_ref:.1f}s')
print(f'α              = {alpha_fitted:.4f}')
print(f'P_novel        = {p_novel_est:.4f}')

## 6. Apply Neg-Trained Model to Pos Data

Same pipeline applied to pos spectra with pos hits — parameters **frozen** from neg fit.
Any drop in performance reflects polarity generalization gap.

In [ ]:
# ── Compute InChIKey for pos hits ──────────────────────────────
if RDKIT_AVAILABLE and 'smiles' in pos_hits_raw.columns:
    pos_hits_raw['ik14']     = pos_hits_raw['smiles'].apply(smiles_to_ik14)
    pos_hits_raw['inchikey'] = pos_hits_raw['smiles'].apply(smiles_to_inchikey)
    with open(INCHIKEY_CACHE, 'w') as f:
        json.dump(ik_cache, f)
    print(f'Pos hits ik14: {pos_hits_raw["ik14"].notna().sum():,}/{len(pos_hits_raw):,}')

# ── Build pos joint (pos has no 'holdout' concept — every TP is test) ──
pos['holdout'] = True  # treat all pos TPs as holdout for evaluation
pos_joint = build_joint(pos, pos_hits_raw)
print(f'Pos joint: {len(pos_joint):,} rows, {pos_joint["wiki_id"].nunique():,} spectra')
print(f'  correct hits: {pos_joint["correct"].sum():,}')

In [ ]:
# ── Filter + sim_gap + priors on pos ──────────────────────────
pos_joint_f = add_sim_gap(pos_joint, MS2_MIN)
pos_joint_f = apply_priors(pos_joint_f)
print(f'Pos after MS2_MIN={MS2_MIN}: {len(pos_joint_f):,} rows, {pos_joint_f["wiki_id"].nunique():,} spectra')

# Pos P_novel (for reference — but we USE the neg-fitted p_novel_est for scoring)
pos_all_wids = set(pos['wiki_id'])
pos_hit_wids = set(pos_joint_f['wiki_id'])
pos_p_novel_obs = len(pos_all_wids - pos_hit_wids) / len(pos_all_wids)
print(f'Pos P_novel (observed, not used): {pos_p_novel_obs:.4f}  vs neg fit: {p_novel_est:.4f}')

In [ ]:
# ── Score pos with frozen neg params ───────────────────────────
pos_scored = score_all(pos_joint_f)
pos_top    = top_calls(pos_scored)
print(f'Pos scored: {pos_scored["wiki_id"].nunique():,} spectra, {len(pos_scored):,} hits')
print(f'Pos top_calls: {len(pos_top):,}')
print(pos_top.groupby(['label','abstain']).size().to_string())

## 7. Performance on Pos

In [ ]:
# ── Top-1 accuracy on pos TPs ─────────────────────────────────
pos_tp_calls = pos_top[pos_top['label']=='TP']
called_tp    = pos_tp_calls[~pos_tp_calls['abstain']]
acc_pos = called_tp['correct'].mean() if len(called_tp) else 0.0
print(f'Pos TP spectra: {len(pos_tp_calls):,}')
print(f'  Called:    {len(called_tp):,} ({len(called_tp)/max(len(pos_tp_calls),1):.1%})')
print(f'  Abstained: {pos_tp_calls["abstain"].sum():,}')
print(f'  Top-1 accuracy on called: {acc_pos:.3f}')
print(f'  Mean posterior (correct):   {called_tp[called_tp["correct"]==True]["post_top"].mean():.3f}')
print(f'  Mean posterior (incorrect): {called_tp[called_tp["correct"]==False]["post_top"].mean():.3f}')

# Baseline: raw entropy_similarity
pos_baseline = (pos_joint_f[pos_joint_f['label']=='TP']
                .sort_values('entropy_similarity', ascending=False)
                .drop_duplicates('wiki_id')['correct'].mean())
print(f'  Baseline (top entropy_similarity): {pos_baseline:.3f}')

# ── FP behavior ────────────────────────────────────────────────
pos_fp_calls = pos_top[pos_top['label']=='FP']
print(f'\nPos FP (yy_) spectra: {len(pos_fp_calls):,}')
print(f'  Abstained: {pos_fp_calls["abstain"].sum():,} ({pos_fp_calls["abstain"].mean():.1%})')

In [ ]:
# ── Compare pos vs neg top-1 accuracy ─────────────────────────
summary = pd.DataFrame({
    'Metric': ['Top-1 accuracy', 'Baseline (raw entropy)', 'Mean post (correct)',
               'Abstention rate', 'Calibration gap'],
    'Neg (holdout)': [f'{acc:.3f}', f'{baseline:.3f}',
                       f'{called[called["correct"]==True]["post_top"].mean():.3f}' if len(called) else 'n/a',
                       f'{test_calls["abstain"].mean():.1%}',
                       '-'],
    'Pos (TPs)':    [f'{acc_pos:.3f}', f'{pos_baseline:.3f}',
                      f'{called_tp[called_tp["correct"]==True]["post_top"].mean():.3f}' if len(called_tp) else 'n/a',
                      f'{pos_tp_calls["abstain"].mean():.1%}',
                      f'{acc_pos - called_tp["post_top"].mean():.3f}' if len(called_tp) else 'n/a'],
    'Delta': [f'{acc_pos-acc:+.3f}', f'{pos_baseline-baseline:+.3f}', '', '', '']
})
print(summary.to_string(index=False))

In [ ]:
# ── Distribution comparison: pos vs neg ───────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# entropy_similarity on correct hits
neg_corr = neg_joint_f[neg_joint_f['correct'] & (neg_joint_f['label']=='TP')]['entropy_similarity']
pos_corr = pos_joint_f[pos_joint_f['correct'] & (pos_joint_f['label']=='TP')]['entropy_similarity']
axes[0,0].hist(neg_corr, bins=40, density=True, alpha=0.5, label=f'Neg (n={len(neg_corr)})', color='steelblue')
axes[0,0].hist(pos_corr, bins=40, density=True, alpha=0.5, label=f'Pos (n={len(pos_corr)})', color='coral')
axes[0,0].set_xlabel('entropy_similarity (correct hits)')
axes[0,0].set_title('MS2 similarity — correct hits'); axes[0,0].legend()

# delta_ppm on correct hits
neg_ppm = neg_joint_f[neg_joint_f['correct']]['delta_ppm'].clip(-10,10)
pos_ppm = pos_joint_f[pos_joint_f['correct']]['delta_ppm'].clip(-10,10)
axes[0,1].hist(neg_ppm, bins=50, density=True, alpha=0.5, label='Neg', color='steelblue')
axes[0,1].hist(pos_ppm, bins=50, density=True, alpha=0.5, label='Pos', color='coral')
axes[0,1].set_xlabel('delta_ppm (correct hits)')
axes[0,1].set_title(f'Mass accuracy — neg σ={neg_ppm.std():.2f}, pos σ={pos_ppm.std():.2f}'); axes[0,1].legend()

# delta_rt on correct hits (annotation source)
neg_rt = neg_joint_f[(neg_joint_f['correct']) & (neg_joint_f['hit_source']=='annotation')]['delta_rt'].dropna().clip(-60,60)
pos_rt = pos_joint_f[(pos_joint_f['correct']) & (pos_joint_f['hit_source']=='annotation')]['delta_rt'].dropna().clip(-60,60)
axes[1,0].hist(neg_rt, bins=40, density=True, alpha=0.5, label=f'Neg (n={len(neg_rt)})', color='steelblue')
axes[1,0].hist(pos_rt, bins=40, density=True, alpha=0.5, label=f'Pos (n={len(pos_rt)})', color='coral')
axes[1,0].set_xlabel('delta_rt annotation (s)')
axes[1,0].set_title('RT deviation — correct hits'); axes[1,0].legend()

# posterior distribution on TPs
axes[1,1].hist(called['post_top'].dropna() if len(called) else [], bins=30, density=True, alpha=0.5, label='Neg called', color='steelblue')
axes[1,1].hist(called_tp['post_top'].dropna() if len(called_tp) else [], bins=30, density=True, alpha=0.5, label='Pos called', color='coral')
axes[1,1].set_xlabel('Posterior of top call (TPs)')
axes[1,1].set_title('Confidence distribution on TPs'); axes[1,1].legend()

plt.tight_layout(); plt.show()

## 8. Export for Oliver

In [ ]:
# ── Build Oliver-friendly Excel output ────────────────────────
pos_spec_meta = pos_raw[[
    'wiki_id','adduct','precursor_mz','rt','entropy',
    'identity_score','fuzzy_score','smiles',
    'annotation-name','annotation-smiles','anno_delta_rt'
]].copy().rename(columns={
    'annotation-name':'anno_name','annotation-smiles':'anno_smiles',
    'smiles':'obs_smiles','adduct':'obs_adduct','precursor_mz':'obs_precursor_mz',
    'rt':'obs_rt','entropy':'obs_entropy'
})

# Top hit info
top_hit = (pos_scored.sort_values(['wiki_id','post'], ascending=[True,False])
           .drop_duplicates('wiki_id')
           [['wiki_id','name','adduct','lib_precursor_mz','smiles','entropy_similarity',
             'sim_gap','delta_ppm','delta_rt','prior_score','LR_gap']]
           .rename(columns={'name':'lib_name_top','adduct':'lib_adduct_top',
                             'lib_precursor_mz':'lib_precursor_mz_top','smiles':'lib_smiles_top',
                             'entropy_similarity':'entropy_similarity_top',
                             'sim_gap':'sim_gap_top','delta_ppm':'delta_ppm_top',
                             'delta_rt':'delta_rt_top','prior_score':'prior_score_top',
                             'LR_gap':'LR_gap_top'}))

# 2nd-best hit
second = (pos_scored.sort_values(['wiki_id','post'], ascending=[True,False])
          .groupby('wiki_id').nth(1).reset_index()
          [['wiki_id','name','entropy_similarity']]
          .rename(columns={'name':'lib_name_2nd','entropy_similarity':'entropy_similarity_2nd'}))

pos_out = pos_top.merge(pos_spec_meta, on='wiki_id', how='left') \
                 .merge(top_hit, on='wiki_id', how='left') \
                 .merge(second, on='wiki_id', how='left')

# Sort by posterior descending (highest-confidence first helps Oliver review)
pos_out = pos_out.sort_values('post_top', ascending=False, na_position='last')

out_xlsx = f'{OUT_DIR}/pos_top_calls.xlsx'
out_csv  = f'{OUT_DIR}/pos_top_calls.csv'
pos_out.to_excel(out_xlsx, index=False)
pos_out.to_csv(out_csv, index=False)
pos_scored.to_csv(f'{OUT_DIR}/pos_assertions.csv', index=False)

print(f'Saved:')
print(f'  {out_xlsx}  ({len(pos_out):,} rows, {len(pos_out.columns)} cols)')
print(f'  {out_csv}')
print(f'  {OUT_DIR}/pos_assertions.csv  ({len(pos_scored):,} rows)')

## 9. Summary Report

Key questions to answer from the numbers above:
1. **Polarity transfer:** Did top-1 accuracy drop significantly (pos vs neg)?
2. **Distributional shift:** Do the σ_M, σ_RT, entropy_sim distributions look similar pos vs neg?
3. **FP handling:** Do pos yy_ flags get captured (low posterior / abstention)? Model has P_novel=0 so shouldn't abstain, but low posterior on FPs is still diagnostic.
4. **Calibration:** Is mean posterior on correct calls close to top-1 accuracy? If yes, model is calibrated on pos. If no, re-fit needed.

If accuracy drop > 5pp or distributions shift substantially, recommend refitting on pos TPs/FPs (requires Oliver's pos curation as labels — which we now have).